epc: exception program counter

pc: program counter

when `ecall` happens:

pc: points to address of trap handler, aka, where to run code next.

epc: points to where execution is cut-off and should be resumed later. It will be written to sepc register and used by sret.

---

simple example, say we have:

scheduler: kernel space, by itself

process A kernel code: it could run some syscall for A, may be cut-off

process A user code

process B kernel code: same idea to A

process B user code

now, when `scheduler()` call `swtch()`, it's switching from what to what?

from scheduler(that's for sure) to either process A or process B kernel code!

you may ask, 3 cases:
1. what if a process just started? there is no kernel code corresponding to it to swtch to!
2. what if a process is running but it never did syscall or anything, again, no kernel code to swtch to!
3. for a process that runs and had kernel doing work for it. make sense.

for 1, it will start with `forkret`, after that, it becomes situation No.2. So the question becomes:
>If a process is running purely in user mode, and never calls the kernel, where does its kernel context come from?

Timer interrupt! Which will goes into kernel code, and this, counts as "kernel work" for process A/B! Even if it's not doing errand for them! It could call `sched()` which is the only path lead back to `schedule()` to swtch again.

---

Question: where is `forkret` being overwritten?

process A runs first time

```
p->context.ra = forkret

scheduler()
    |
    | swtch()
    v
forkret()
    |
    v
usertrapret() → sret
    |
    v
User A
```

right now, p->context.ra is still forkret.

Now, timer interrupt happens.

```
User A (already runs for a while)
    |
    | Timer interrupt
    v
uservec()
    |
    | Save user ra into trapframe->ra
    v
usertrap()
    |
    v
yield()
    |
    v
sched()
    |
    | swtch(&p->context, &mycpu()->context)
    v
scheduler()
```

p->context.ra is overwritten in swtch()!

ra is sched() call swtch() return add, managed by compiler.

```c
intena = mycpu()->intena;
swtch(&p->context, &mycpu()->context); -> swtich out here
mycpu()->intena = intena;
```

so kernel will switch out at `swtch` and ra is the address next line (nor forkret).

`mycpu()->intena = intena;` will not run, it is suspended work, just like suspended unfinished syscall work.

---

question: how (ex. after timer interrupt) to go from kernel to user process?

continue from above, say the kernel switch back the process A's kernel code, it will continue from suspended `swtch()`

```c
if (which_dev == 2)
    yield();

  prepare_return();

 // the user page table to switch to, for trampoline.S
  uint64 satp = MAKE_SATP(p->pagetable);

  // return to trampoline.S; satp value in a0.
  return satp;
```

why it returns user's satp? because the caller is uservec, it just continues the execution.

```asm
uservec:    
    	... 
        
        jalr t0

.globl userret
userret:
        ...
        
        csrw satp, a0
```

`jalr t0` will call usertrap, which will store satp at a0, and then, after usertrap is done, `jalr` will make pc continues to execute `userret`! which will use a0 and do `csrw satp, a0` and eventually sret to jump to user process.

---

When timer interrupt happens

#### cpu runs scheduler

`myproc() == 0, no yield()`

scheduler() resumes, as if nothing happens.

Scheduler doesn't need timer preemption, but to make sleeping processes runnable it's needed here.

#### cpu runs user code

usertrap() → yield() → sched(). Later resumes from kernel and returns to user.

#### cpu runs kernel code

kerneltrap() → possibly yield() → sched() or doing syscall work. All cases go back to user.


kernelvec situation

| CPU Executing | Timer Interrupt | Exception |
|---|---|---|
| `scheduler()` | Handle → `sret` → resume scheduler | `panic` |
| Process A: `syscall()` | `yield()` → `sched()` → later resume syscall | `panic` |
| Process A: `yield()` / `sched()` | Interrupts disabled; cannot be interrupted by the timer | `panic` |

uservec situation


| CPU Executing | Syscall (`ecall`) | Timer Interrupt | Other Device Interrupt | Exception (page fault, illegal instruction, etc.) |
|---|---|---|---|---|
| Process A: user code | `syscall()` → `sret` → resume user | `yield()` → `sched()` → later resume user | Handle → `sret` → resume user | Kill process |
| Process B: user code | Same as A | Same as A | Same as A | Same as A |

**Key Points:**

- `ecall`: `sepc += 4` to avoid repeating the syscall.
- Other device interrupts: Usually return directly to User Mode after being handled.
- Unexpected exceptions: xv6 typically marks the process as killed, then calls `exit()` instead of returning normally to User Mode.

**In User Mode, we do not need to distinguish between `scheduler()`, `syscall()`, and `sched()` as we do in Kernel Mode, because all three can only execute in Kernel Mode.**

**Scheduler always switch process's kernel context**